# Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelBinarizer

# Definição de caminhos

In [ ]:
# --- Caminhos para os arquivos ---
TRAIN_FILE_PATH = '../data/extracted_features/train_features_completas.csv'
TEST_FILE_PATH = '../data/extracted_features/test_features_completas.csv'

# --- Carregar os dados ---
try:
    train_df = pd.read_csv(TRAIN_FILE_PATH)
    test_df = pd.read_csv(TEST_FILE_PATH)
    print(f"Arquivos carregados com sucesso!")
    print(f"Formato dos dados de treino: {train_df.shape}")
    print(f"Formato dos dados de teste:  {test_df.shape}")
except FileNotFoundError:
    print(f"Erro: Arquivos não encontrados.")
    print(f"Verifique se os caminhos '{TRAIN_FILE_PATH}' e '{TEST_FILE_PATH}' estão corretos.")
    print("Se estiver no Google Colab, certifique-se de que os arquivos foram enviados e o caminho está correto.")

train_df = train_df.dropna(subset=['Classe'])
test_df = test_df.dropna(subset=['Classe'])


print("\n--- Amostra dos Dados de Treino ---")
display(train_df.head())

# Preparação dos dados

In [ ]:
# 1. Separar features (X) e labels (y)
# Primeiro, separamos os labels
y_train = train_df['Classe']
y_test = test_df['Classe']

# Em seguida, selecionamos apenas as colunas de features (k-mers)
# Adicione 'Classe' à lista de colunas a serem descartadas
feature_names = train_df.drop(columns=['nameseq', 'label', 'Sequência de TE', 'Classe']).columns
X_train = train_df[feature_names]
X_test = test_df[feature_names]

print(f"Número de features: {len(feature_names)}")
print(f"Exemplo de features: {feature_names[:5].to_list()}...")

# 2. FORÇAR A CONVERSÃO DE X PARA NUMÉRICO (A CORREÇÃO)
# O erro 'ValueError' indica que há strings ('AAA') nas colunas de features.
# 'errors='coerce'' transformará essas strings problemáticas em 'NaN' (Not a Number).
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# 3. Verificar se a coerção criou valores NaN (o que indica dados "sujos")
nan_in_train = X_train.isna().sum().sum()
if nan_in_train > 0:
    print(f"\nAlerta: {nan_in_train} valores não numéricos (strings) foram encontrados")
    print("nas colunas de features do treino e convertidos para NaN.")
    
    # Estratégia de Pré-processamento: Preencher NaNs com a média da coluna.
    # Isso permite que o StandardScaler funcione.
    X_train = X_train.fillna(X_train.mean())
    print("Valores NaN foram preenchidos com a média da sua respectiva coluna.")
else:
    print("\nColunas de features do treino parecem ser 100% numéricas.")

# 4. Aplicar o mesmo para o conjunto de teste
nan_in_test = X_test.isna().sum().sum()
if nan_in_test > 0:
    print(f"Alerta: {nan_in_test} valores não numéricos encontrados no teste.")
    # IMPORTANTE: Preencher NaNs do teste com a média do TREINO
    X_test = X_test.fillna(X_train.mean()) 
    print("Valores NaN do teste foram preenchidos com a média do TREINO.")

# 5. Aplicar LabelEncoder nos labels (y)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"\nClasses originais: {le.classes_}")
print(f"Classes codificadas: {np.unique(y_train_encoded)}")

In [ ]:

# 3. Aplicar StandardScaler nas features (X)
scaler = StandardScaler()

# Ajustar o scaler com os dados de TREINO
X_train_scaled = scaler.fit_transform(X_train)

# Apenas transformar os dados de TESTE (usando o ajuste do treino)
X_test_scaled = scaler.transform(X_test)

print(f"Shape dos dados de treino escalados: {X_train_scaled.shape}")

# Validação cruzada para seleção de hiperparâmetros

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_dist_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    random_state=42,
    tree_method='hist',   # novo padrão
    device='cuda',        # força uso da GPU
    eval_metric='logloss'
)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=10,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1,   # CRÍTICO para não travar WSL
    scoring='accuracy'
)

random_search_xgb.fit(X_train, y_train_encoded)
print(f"Melhores hiperparâmetros XGBoost: {random_search_xgb.best_params_}")


Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=  14.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   9.8s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=  10.0s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=200, subsample=1.0; total time=  19.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=200, subsample=1.0; total time=  19.2s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=200, subsample=1.0; total time=  17.7s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time=   5.1s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=0.8; total time=   5.0s
[CV] END colsample_

In [10]:
# Importar as bibliotecas necessárias
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# random_search_xgb já foi treinado na célula anterior
# 1. Instanciar o classificador com os melhores parâmetros encontrados
best_params_from_search = random_search_xgb.best_params_
print(f"Usando os melhores hiperparâmetros da busca: {best_params_from_search}")

final_xgb_model = xgb.XGBClassifier(
    **best_params_from_search,  # Desempacota os melhores parâmetros aqui
    random_state=42,
    tree_method='hist',
    device='cuda',
    eval_metric='mlogloss',
    use_label_encoder=False
)

# 2. Treinar o modelo final com os dados de treino completos
print("\nIniciando o treinamento do modelo final...")
# Nota: É uma boa prática retreinar o modelo com os melhores parâmetros em todo o conjunto de treino
final_xgb_model.fit(X_train_scaled, y_train_encoded)
print("Treinamento concluído.")

# 3. Fazer previsões e avaliar
print("\nRealizando previsões no conjunto de teste...")
y_pred_final = final_xgb_model.predict(X_test_scaled)
print("Previsões concluídas.")

# Recrie o LabelEncoder e ajuste-o aos dados de treino originais para recuperar os nomes das classes.
label_encoder = LabelEncoder()
label_encoder.fit(y_train_encoded)

accuracy_final = accuracy_score(y_test_encoded, y_pred_final)
print(f'\nAcurácia do XGBoost final: {accuracy_final:.4f}')
print('\nRelatório de Classificação:\n')
print(classification_report(y_test_encoded, y_pred_final, target_names=label_encoder.classes_))


Usando os melhores hiperparâmetros da busca: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

Iniciando o treinamento do modelo final...


/home/gabyl/projetos/trabalho-TEsClassification/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:53:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Treinamento concluído.

Realizando previsões no conjunto de teste...
Previsões concluídas.

Acurácia do XGBoost final: 0.5891

Relatório de Classificação:



TypeError: object of type 'numpy.int64' has no len()